# 🎲 Group B — The Coin-Flip Model (Stochastic)

*You will measure this outbreak's R₀ by simulating whole people and real
luck — hundreds of random epidemics — and then find the vaccination level
that makes outbreaks improbable.*

### How to read this notebook

| Marker | What to do |
|---|---|
| 📖 **IDEA** | Read the explanation first |
| ✏️ **EDIT ME** | Change a value, re-run, and see what moves |
| ▶️ **RUN** | Run the code cell |
| 👀 **READ** | Inspect the result, graph, or message |
| 🧠 **BUILD IT** | A core concept turned into code — read this one closely |
| ✅ **CHECKPOINT** | Pause and answer the questions |
| 🔒 **RUN ONLY** | Infrastructure. Run it; you do not need to memorize it |
| ⭐ **OPTIONAL** | Try only after the required work is complete |

> 🖥️ **Before anything else — pick the kernel.** In the top-right corner of
> Jupyter, the kernel should read **`Python (epidemic-modeling)`**. Your site
> may name it differently — for example `Python (<site_kernel_name>)` — so if
> you don't see it, ask your project lead *before* debugging anything.
> A wrong kernel makes the setup cell fail with `ModuleNotFoundError`.

> **Never written Python before? That is expected here.** Follow the markers,
> read the “What the next cell does” notes, and connect each graph back to the
> question. If the code itself feels unfamiliar, the **master notebook at the
> repository root** opens with a 🧭 *Python survival guide* and a 🧩 *function
> map* of every tool — keep it open in another tab.

## Project path

The whole project — including the presentation and practice — fits within
**10–12 hours**.

> **Shared question** *(when does an outbreak explode, and how much
> vaccination stops it?)*
> ➜ **Group B: random individual infections (whole people, luck included)**
> ➜ **Shared measurement** *(this outbreak's R₀)*
> ➜ **Shared experiment** *(the vaccination threshold)*
> ➜ **Group presentation** *(equations vs coin flips — do they agree?)*

## The research question

> **When does an outbreak explode — and how much vaccination stops it?**

A disease has swept through a town of **10,000 people**. All we have is the
public-health record: how many people got sick each day, for 150 days
(`data/observed_outbreak.csv`). You are the disease detectives.

Epidemiologists describe outbreaks with three groups of people and one number:

| Symbol | Who they are |
|---|---|
| **S** — Susceptible | could still catch it |
| **I** — Infectious | sick now, and spreading it |
| **R** — Recovered | had it, now immune |

**R₀ ("R-naught")** = how many people one sick person infects, on average,
when everyone around them is susceptible. If R₀ > 1 the outbreak grows; if
R₀ < 1 it dies out. Your two jobs:

1. **Measure this outbreak's R₀** from the daily-case record.
2. **Find the vaccination level that would have prevented it** — and compare
   your answer with the famous formula for herd immunity, **v\* = 1 − 1/R₀**.

## Setup — run this first

### What the next cell does 📖

1. **Imports the toolbox.** Think of an import as: *“Python, please give me
   this toolbox.”* — `numpy` does math on whole lists of numbers at once,
   `matplotlib` draws graphs.
2. **Loads the outbreak record** from `data/observed_outbreak.csv` into an
   array called `observed` — one number per day: how many people got sick.
3. **Loads three small helper tools** (explained right above their use) and
   **prints a check** so you know everything is ready.

In [ ]:
import sys
sys.path.insert(0, "..")   # helper tools live at the repository root
import numpy as np
import matplotlib.pyplot as plt

# three small tested tools (🔒 in epidemic_helpers.py — see the master's 🧩 map):
#   align_to_threshold : shift a curve so day 0 = the day it reached 20 cases
#   rmse               : the typical difference between two curves
#   analytic_final_size: textbook prediction of the outbreak's final size
from epidemic_helpers import align_to_threshold, rmse, analytic_final_size

observed = np.loadtxt("../data/observed_outbreak.csv",
                      delimiter=",", skiprows=1)[:, 1]

N = 10_000          # people in the town
GAMMA = 0.25        # recovery rate: 1 / (average 4 days infectious)

print(f"days of records: {len(observed)}")
print(f"total people infected: {int(observed.sum()) + 5} of {N}")
print("Preflight OK — you are ready to start.")

## 👀 Meet the outbreak

### What the next cell does 📖

Two pictures of the epidemic: **left** — how many people got sick each day
(the famous “epidemic curve”); **right** — the running total. Steps: plot the
daily numbers, add them up with `np.cumsum`, plot the total, label everything.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
days = np.arange(len(observed))

axes[0].bar(days, observed, width=1.0, color="C3", alpha=0.8)
axes[0].set_xlabel("day"); axes[0].set_ylabel("new cases per day")
axes[0].set_title("The outbreak, day by day")

axes[1].plot(days, np.cumsum(observed), color="C3")
axes[1].set_xlabel("day"); axes[1].set_ylabel("total people infected")
axes[1].set_title("The running total")

for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

### ✅ CHECKPOINT — data

1. Around which day did the outbreak peak? Roughly how many fell sick that day?
2. What fraction of the town was eventually infected?
3. Before any model: why do outbreaks *stop on their own*, even with nobody
   vaccinated? (Hint: who is left to infect?)

### ✏️ EDIT ME — the only settings cell

Run the whole notebook once with these defaults. Then come back, change
**one** value, and re-run everything below this cell.

In [ ]:
# ✏️ PARTICIPANT EDIT AREA — change values here, nowhere else
R0_CANDIDATES = [2.0, 2.2, 2.5, 2.8, 3.0]
VACCINATION_LIST = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.55, 0.6, 0.65, 0.7, 0.8]
N_SIMULATIONS = 200          # epidemics per spaghetti plot

print(f"{len(R0_CANDIDATES)} candidate R0 values, "
      f"{N_SIMULATIONS} simulations per experiment")

## 🧠 BUILD IT — the coin-flip epidemic: whole people, real randomness

Group A's equations track *average* behavior — 12.7 infections per day is a
fine average, but no real town infects 0.7 of a person. Your model tracks
**whole people and luck**: each day, every susceptible person either escapes
infection or doesn't — decided by a (weighted) coin flip.

### The Python, decoded 🧩

| Line | What it does in plain English |
|---|---|
| `rng = np.random.default_rng()` | A random-number generator — the coin we flip. |
| `1 - np.exp(-beta * I / N)` | Today's chance that one susceptible person gets infected. |
| `rng.binomial(S, p)` | Flip that weighted coin for **all S people at once**; count the infections. |
| `rng.binomial(I, GAMMA)` | Each sick person recovers today with chance `GAMMA`; count who does. |
| `if I == 0: break` | No one left infectious → the outbreak is over, stop early. |

In [ ]:
def simulate_outbreak(R0, vaccinated_frac=0.0, days=150, I0=5, rng=None):
    """One RANDOM epidemic — a different story every time you run it."""
    if rng is None:
        rng = np.random.default_rng()       # unseeded = truly random
    beta = R0 * GAMMA
    S = int(N * (1 - vaccinated_frac)) - I0
    I = I0
    new_cases = np.zeros(days)
    for day in range(days):
        p = 1 - np.exp(-beta * I / N)        # today's infection risk
        infections = rng.binomial(S, p)      # coin flips for all S people
        recoveries = rng.binomial(I, GAMMA)  # coin flips for all I people
        S -= infections
        I += infections - recoveries
        new_cases[day] = infections
        if I == 0:
            break                            # outbreak over
    return new_cases

print("stochastic model defined — run the NEXT cell twice and compare!")

### What the next cell does 📖

Runs **one** random epidemic at R₀ = 2.5 and plots it against the observed
outbreak.

**Now the important part: run this cell two or three times** (click it, press
Shift+Enter, repeat). Same disease, same town, same R₀ — different epidemic
every time. *That* is what the equations can't show you.

In [ ]:
one_run = align_to_threshold(simulate_outbreak(2.5), 20)
obs_aligned = align_to_threshold(observed, 20)

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.bar(range(len(obs_aligned)), obs_aligned, width=1.0, color="C3",
       alpha=0.5, label="observed outbreak")
ax.plot(one_run, color="C0", lw=2, label="one random epidemic (R0 = 2.5)")
ax.set_xlabel("days since 20 cases"); ax.set_ylabel("new cases per day")
ax.legend(); ax.grid(alpha=0.3)
ax.set_title("Re-run me! I am different every time")
plt.tight_layout(); plt.show()

## 👀 One run means nothing — run two hundred

With randomness, a single simulation is an anecdote. The honest move is to
run **many** and look at the whole cloud.

### What the next cell does 📖

1. runs `N_SIMULATIONS` random epidemics at R₀ = 2.5 (a small loop),
2. draws every one as a faint line — a “spaghetti plot”,
3. lays the observed outbreak on top.

**Predict before you run:** will reality sit inside the spaghetti, or outside it?

In [ ]:
rng = np.random.default_rng(0)          # seeded, so everyone gets the same cloud

fig, ax = plt.subplots(figsize=(8.5, 4))
final_sizes = []
for i in range(N_SIMULATIONS):
    run = simulate_outbreak(2.5, rng=rng)
    final_sizes.append(run.sum())
    aligned = align_to_threshold(run, 20)
    if len(aligned) > 0:
        ax.plot(aligned, color="C0", alpha=0.08)
ax.bar(range(len(obs_aligned)), obs_aligned, width=1.0, color="C3",
       alpha=0.55, label="observed outbreak")
ax.set_xlabel("days since 20 cases"); ax.set_ylabel("new cases per day")
ax.set_title(f"{N_SIMULATIONS} possible epidemics, one reality")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f"epidemics that fizzled early (< 500 total cases): "
      f"{int(np.sum(np.array(final_sizes) < 500))} of {N_SIMULATIONS}")

## 🧪 Measure R₀ — which candidate's cloud contains reality?

Your version of fitting: for each candidate R₀, run many epidemics, take the
**middle curve** (the median), and ask which candidate's typical epidemic
looks most like the real one.

### What the next cell does 📖

1. loops over your `R0_CANDIDATES`,
2. runs 100 epidemics per candidate (🔒 keeping only the ones that took off),
3. computes each candidate's median curve and its difference from reality,
4. prints the winner.

In [ ]:
def median_curve(R0, n_runs=100, seed=2):
    """The 'typical' epidemic at this R0: median of many aligned runs."""
    rng = np.random.default_rng(seed)
    aligned_runs = []
    for _ in range(n_runs):
        run = simulate_outbreak(R0, rng=rng)
        if run.sum() > 1000:                     # keep runs that took off
            aligned_runs.append(align_to_threshold(run, 20))
    if not aligned_runs:                         # nothing took off at this R0
        return np.zeros_like(obs_aligned)
    length = min(len(a) for a in aligned_runs)
    return np.median([a[:length] for a in aligned_runs], axis=0)

errors = []
for R0 in R0_CANDIDATES:
    errors.append(rmse(obs_aligned, median_curve(R0)))
    print(f"R0 = {R0:<4}  difference from reality = {errors[-1]:6.1f} cases/day")

best_R0 = R0_CANDIDATES[int(np.argmin(errors))]
print(f"\nYour measurement: this outbreak's R0 ≈ {best_R0}")
print(f"Herd-immunity formula predicts the threshold: 1 - 1/R0 = {1 - 1/best_R0:.2f}")

### ✅ CHECKPOINT — result

1. Write down your R₀ measurement — you will present it.
2. Some of your 200 spaghetti epidemics fizzled after a handful of cases —
   with the *same* R₀ > 1. How is that possible? (Think about the first few
   coin flips.)
3. Group A fits one smooth curve and gets one number. What extra information
   does your cloud give that their curve cannot?

## 🧪 The luck experiment — one sick traveler arrives

Equations say: R₀ = 2.5 > 1, so an outbreak *must* grow. Coin flips disagree:
if the **first** sick person happens to recover before infecting anyone, the
outbreak dies at one case. Theory says that lucky escape happens with
probability about **1/R₀ = 0.4**.

### What the next cell does 📖

1. runs 500 epidemics that each start from **a single case** (`I0=1`),
2. counts how many fizzle out (fewer than 500 total cases),
3. compares that fraction with the 1/R₀ prediction.

In [ ]:
rng = np.random.default_rng(4)
fizzled = 0
for _ in range(500):
    run = simulate_outbreak(2.5, I0=1, rng=rng)
    if run.sum() < 500:
        fizzled += 1

print(f"outbreaks that fizzled: {fizzled} of 500  ({fizzled/500:.0%})")
print(f"theory's prediction:    about 1/R0 = {1/2.5:.0%}")
print("\nSame town, same disease — sometimes the whole outbreak")
print("comes down to a few coin flips at the start.")

## 🧪 The shared experiment — how much vaccination stops it?

Same design as Group A, but with randomness the honest question changes from
*“how big is the outbreak?”* to *“what is the **probability** of an outbreak?”*

### What the next cell does 📖

1. loops over your `VACCINATION_LIST`,
2. runs 100 random epidemics per vaccination level (your fitted R₀),
3. plots the fraction that became real outbreaks (> 500 cases), with the
   herd-immunity formula as a dashed line.

**Predict before you run:** at exactly the threshold, is the outbreak
probability 0, 1, or something in between?

In [ ]:
rng = np.random.default_rng(3)
outbreak_prob = []
for v in VACCINATION_LIST:
    outbreaks = 0
    for _ in range(100):
        run = simulate_outbreak(best_R0, vaccinated_frac=v, rng=rng)
        if run.sum() > 500:
            outbreaks += 1
    outbreak_prob.append(outbreaks / 100)

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.plot(VACCINATION_LIST, outbreak_prob, "o-", color="C2")
ax.axvline(1 - 1/best_R0, color="k", ls="--",
           label=f"herd-immunity formula: 1 - 1/R0 = {1 - 1/best_R0:.2f}")
ax.set_xlabel("fraction of the town vaccinated on day 0")
ax.set_ylabel("probability of a real outbreak")
ax.set_title("The vaccination cliff — probabilistic edition")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

### ✅ CHECKPOINT — conclusion

1. Where does the outbreak probability collapse? How close to 1 − 1/R₀?
2. Near the threshold the answer is neither 0 nor 1 — luck decides. Say in
   one sentence what that means for real public-health decisions.
3. Group A's equations give a sharp cliff; your coin flips give a *soft*
   cliff. Which picture do you find more honest, and why?

## 📊 Variable and column reference

Use this table to make **your own extra graphs** without guessing what
names mean. Everything listed is in memory after running the notebook.

| Name | What it is | Good for plotting |
|---|---|---|
| `observed` | array, 150 days of new cases | the epidemic curve |
| `obs_aligned` | the same curve, day 0 = 20th case | comparisons with models |
| `N`, `GAMMA` | town size (10,000) and recovery rate (0.25/day) | — |
| `best_R0` | your measurement of this outbreak's R₀ | headline number |
| `R0_CANDIDATES`, `VACCINATION_LIST` | your ✏️ experiment settings | axes |
| `errors` | difference-from-reality per candidate | the fitting dip |
| `simulate_outbreak(R0, vaccinated_frac)` | your random model — new every call | any what-if |
| `median_curve(R0)` | the typical epidemic at a given R₀ | overlays |
| `outbreak_prob` | outbreak probability per vaccination level | the soft cliff |

*Example:* `plt.plot(np.cumsum(observed))` — the outbreak's running total.

## ✅ CHECKPOINT — Group B presentation

Your talk should answer, with evidence:

- **Shared research question:** when does an outbreak explode, and how
  much vaccination stops it?
- **Group B choice:** a stochastic model — whole people, coin flips, luck included
- **Shared measurement:** this outbreak's R₀ ≈ [your number]
- **Shared experiment:** the vaccination cliff — where it falls, vs the
  formula 1 − 1/R₀
- **Your method's special insight:** the *soft* cliff and stochastic extinction: near the threshold —
  and at every outbreak's start — luck decides
- **Conclusion:** [one or two evidence-based sentences]
- **Limitation and next question:** [e.g., everyone mixes equally here —
  no households, schools, or superspreaders; what would a network change?]

Presentation preparation and practice fit **inside** the program's
10–12 hour total. Compare R₀ and thresholds with the other group — do
equations and coin flips agree?

## ⭐ OPTIONAL — exploration

1. **Histogram of fates.** Plot `plt.hist(final_sizes, bins=40)` from the
   spaghetti cell — outbreaks are either tiny or huge, almost never medium.
   Why?
2. **More travelers.** Repeat the luck experiment with `I0 = 2, 3, 5`. How
   fast does the fizzle probability shrink? (Theory: (1/R₀)^I₀.)
3. **A smaller town.** Set `N = 1000`. Does randomness matter more or less?